In [3]:
from pathlib import Path
import json
import os
import pandas as pd
import plotly.express as px

In [9]:
BASE_DIR = Path(os.getcwd())
DETECTED_ANOMALY_FILE = BASE_DIR / "detected_anomaly.parquet"

In [10]:
def load_detected_anomaly() -> pd.DataFrame:
    if not DETECTED_ANOMALY_FILE.exists():
        raise FileNotFoundError(f"Missing file: {DETECTED_ANOMALY_FILE}")

    df = pd.read_parquet(DETECTED_ANOMALY_FILE)
    df.columns = [col.lower() for col in df.columns]
    return df

In [24]:
def load_cleaned_budget_for_year(year: int) -> pd.DataFrame:
    parquet_file = BASE_DIR / f"cleaned_budget_{year}.parquet"
    if not parquet_file.exists():
        raise FileNotFoundError(f"Missing file: {parquet_file}")

    df = pd.read_parquet(parquet_file)
    df.columns = [col.lower() for col in df.columns]
    return df

In [11]:
anomaly_df = load_detected_anomaly()

In [17]:
yearly_anomalies = anomaly_df.groupby("fiscal_year").size().reset_index(name="anomaly_count")

In [21]:
anomaly_df

,fiscal_year,department_code,department_name,full_agency_code,agency_name,region_code,region_description,uacs_object_code,uacs_sub_object_name,budget_amount_nep,budget_amount_gaa,unapproved_budget,inserted_budget,abs_change,pct_change,adjustment_type,anomaly_threshold,z_score,anomaly_zscore,region_mean,region_std,region_anomaly,historical_mean,historical_std,historical_anomaly,anomaly_score,is_anomaly,explanation
0,2020,01,Congress of the Philippines (CONGRESS),01001,Senate,13,National Capital Region (NCR),5010101001,Basic Salary - Civilian,1385509.0,1385509.0,False,False,0.0,0.0,No Significant Change,False,-0.032767,False,259.337562,7918.851794,False,0.000000,0.000000,False,0,False,
1,2020,01,Congress of the Philippines (CONGRESS),01001,Senate,13,National Capital Region (NCR),5010102000,Salaries and Wages - Casual/Contractual,60383.0,60383.0,False,False,0.0,0.0,No Significant Change,False,-0.051481,False,0.432426,8.413488,False,-0.091343,0.409758,False,0,False,
2,2020,01,Congress of the Philippines (CONGRESS),01001,Senate,13,National Capital Region (NCR),5010201001,PERA - Civilian,46272.0,46272.0,False,False,0.0,0.0,No Significant Change,False,-0.032705,False,25.517430,780.644898,False,0.001470,0.003600,False,0,False,
3,2020,01,Congress of the Philippines (CONGRESS),01001,Senate,13,National Capital Region (NCR),5010202000,Representation Allowance (RA),31086.0,31086.0,False,False,0.0,0.0,No Significant Change,False,-0.025357,False,0.152793,6.028964,False,-0.141838,0.378423,False,0,False,
4,2020,01,Congress of the Philippines (CONGRESS),01001,Senate,13,National Capital Region (NCR),5010203001,Transportation Allowance (TA),31086.0,31086.0,False,False,0.0,0.0,No Significant Change,False,-0.025357,False,0.152790,6.028965,False,0.001193,0.002923,False,0,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354336,2026,40,Department of Migrant Workers (DMW),40002,Overseas Workers Welfare Administration,13,National Capital Region (NCR),5029902000,Printing and Publication Expenses,1228.0,0.0,True,False,-1228.0,-1.0,Unapproved Budget,True,-0.187990,False,-0.965866,0.181688,False,-0.250000,0.500000,False,1,False,Flagged due to: large budget change
354337,2026,40,Department of Migrant Workers (DMW),40002,Overseas Workers Welfare Administration,13,National Capital Region (NCR),5029903000,Representation Expenses,9678.0,0.0,True,False,-9678.0,-1.0,Unapproved Budget,True,-0.161451,False,-0.974596,0.157440,False,-0.250000,0.500000,False,1,False,Flagged due to: large budget change
354338,2026,40,Department of Migrant Workers (DMW),40002,Overseas Workers Welfare Administration,13,National Capital Region (NCR),5029904000,Transportation and Delivery Expenses,5770.0,0.0,True,False,-5770.0,-1.0,Unapproved Budget,True,-0.291514,False,-0.921676,0.268926,False,-0.250000,0.500000,False,1,False,Flagged due to: large budget change
354339,2026,40,Department of Migrant Workers (DMW),40002,Overseas Workers Welfare Administration,13,National Capital Region (NCR),5029922000,Bank Transaction Fee,8475.0,0.0,True,False,-8475.0,-1.0,Unapproved Budget,True,-0.301511,False,-0.916667,0.277836,False,-0.333333,0.577350,False,1,False,Flagged due to: large budget change


In [23]:
anomaly_df.groupby(["department_code", "department_name"]).agg(
    {"budget_amount_nep": "sum", "budget_amount_gaa": "sum"}
).reset_index().sort_values(by="budget_amount_gaa", ascending=False).head(10)

,department_code,department_name,budget_amount_nep,budget_amount_gaa
17,18,Department of Public Works and Highways (DPWH),4.331142e+09,5.044799e+09
6,07,Department of Education (DepEd),4.243126e+09,4.048367e+09
13,14,Department of the Interior and Local Governmen...,1.557480e+09,1.540868e+09
16,17,Department of National Defense (DND),1.383707e+09,1.342079e+09
19,20,Department of Social Welfare and Development (...,1.371722e+09,1.210721e+09
12,13,Department of Health (DOH),1.149627e+09,1.146307e+09
33,35,Budgetary Support to Government Corporations (...,1.102313e+09,1.064937e+09
7,08,State Universities and Colleges (SUCs),5.941006e+08,6.474521e+08
4,05,Department of Agriculture (DA),6.222902e+08,5.476596e+08
36,38,Department of Transportation (DOTr),1.007322e+09,4.327562e+08


In [26]:
budget_raw = load_cleaned_budget_for_year(2020)

In [30]:
anomaly_df[anomaly_df['is_anomaly']== True].iloc[0]

fiscal_year                                                          2020
department_code                                                        01
department_name                    Congress of the Philippines (CONGRESS)
full_agency_code                                                    01001
agency_name                                                        Senate
region_code                                                            13
region_description                          National Capital Region (NCR)
uacs_object_code                                               5021003000
uacs_sub_object_name             Extraordinary and Miscellaneous Expenses
budget_amount_nep                                                185442.0
budget_amount_gaa                                                260442.0
unapproved_budget                                                   False
inserted_budget                                                     False
abs_change                            

In [31]:
anomaly_profile = anomaly_df[anomaly_df['is_anomaly']== True].iloc[0]

In [59]:
show_budget_profile = budget_raw[
    (budget_raw["uacs_object_code"] == anomaly_profile["uacs_object_code"])
    & (budget_raw["department_code"] == anomaly_profile["department_code"])
    & (budget_raw["fiscal_year"] == anomaly_profile["fiscal_year"])
    & (budget_raw["region_code"] == anomaly_profile["region_code"])
]

In [60]:
show_budget_profile

,budget_id,budget_type,fiscal_year,budget_amount,budget_description,funding_code,funding_source,department_code,department_name,abbreviation,agency_code,full_agency_code,agency_name,org_code,org_name,region_code,region_description,prexc_fpap_id,uacs_object_code,uacs_classification,uacs_sub_class,uacs_group,uacs_object_name,uacs_sub_object_name
46,GAA-2020-0000000047,GAA,2020,99496.0,General management and supervision,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,001,01001,Senate,010010000000,Senate,13,National Capital Region (NCR),100000100001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
104,GAA-2020-0000000105,GAA,2020,0.0,Senate Relocation,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,001,01001,Senate,010010000000,Senate,13,National Capital Region (NCR),100000200001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
138,GAA-2020-0000000139,GAA,2020,160946.0,Legislation of Laws and Other Related Activities,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,001,01001,Senate,010010000000,Senate,13,National Capital Region (NCR),310100100001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
189,GAA-2020-0000000190,GAA,2020,400.0,General management and supervision,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,002,01002,Senate Electoral Tribunal,010020000000,Senate Electoral Tribunal,13,National Capital Region (NCR),100000100001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
245,GAA-2020-0000000246,GAA,2020,4800.0,Adjudication of Electoral Contests involving M...,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,002,01002,Senate Electoral Tribunal,010020000000,Senate Electoral Tribunal,13,National Capital Region (NCR),310100100001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
314,GAA-2020-0000000315,GAA,2020,5958.0,General management and supervision,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,003,01003,Commission on Appointments,010030000000,Commission on Appointments,13,National Capital Region (NCR),100000100001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
397,GAA-2020-0000000398,GAA,2020,536240.0,General management and supervision,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,004,01004,House of Representatives,010040000000,House of Representatives,13,National Capital Region (NCR),100000100001000,5021003000,Expenses,Maintenance and Other Operating Expenses,"Confidential, Intelligence and Extraordinary E...",Extraordinary and Miscellaneous Expenses,Extraordinary and Miscellaneous Expenses
443,GAA-2020-0000000444,GAA,2020,595240.0,Legislation of laws and other related activities,01101101,Regular Agency Fund - General Fund - New Gener...,01,Congress of the Philippines (CONGRESS),CONGRESS,004,01004,House of Represen

In [61]:
even_budget_details = pd.merge(show_budget_profile[show_budget_profile['budget_type']== 'NEP'], show_budget_profile[show_budget_profile['budget_type']== 'GAA'],
         on=['fiscal_year','org_code','budget_description','region_code'], how='outer',suffixes=('_nep', '_gaa')).where(lambda x: x['budget_amount_nep'] != x['budget_amount_gaa'])

In [62]:
even_budget_details[['budget_description','budget_amount_nep', 'budget_amount_gaa','funding_source_nep','department_name_nep','agency_name_nep']
                    ]

,budget_description,budget_amount_nep,budget_amount_gaa,funding_source_nep,department_name_nep,agency_name_nep
0,General management and supervision,44496.0,99496.0,Regular Agency Fund - General Fund - New Gener...,Congress of the Philippines (CONGRESS),Senate
1,Legislation of Laws and Other Related Activities,140946.0,160946.0,Regular Agency Fund - General Fund - New Gener...,Congress of the Philippines (CONGRESS),Senate
2,Senate Relocation,NaN,0.0,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
5,General management and supervision,5472.0,5958.0,Regular Agency Fund - General Fund - New Gener...,Congress of the Philippines (CONGRESS),Commission on Appointments
6,General management and supervision,470000.0,536240.0,Regular Agency Fund - General Fund - New Gener...,Congress of the Philippines (CONGRESS),House of Representatives
7,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN
